In [ ]:
import os, sys, json, warnings, time
import numpy as np
import pandas as pd
import joblib
warnings.filterwarnings("ignore")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

BASE = "C:/Bastion_IDS"
sys.path.insert(0, BASE)
NB_OUT = os.path.join(BASE, "notebooks", "unsw_only_retrain", "outputs")
RESULTS_OUT = os.path.join(NB_OUT, "results")
os.makedirs(RESULTS_OUT, exist_ok=True)

from core.engine import BastionEngine
from tensorflow import keras
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix)
from scipy import stats

In [ ]:
# ------------------------------------------------------------
# 1. LOAD REAL TEST SET (same 175,341-row file every other Day-1/A1 result uses)
# ------------------------------------------------------------
eng = BastionEngine()
print("production rf:", type(eng.rf).__name__, "| autoencoder loaded:", eng.autoencoder is not None)

df = pd.read_csv(os.path.join(BASE, "data", "UNSW_NB15_testing-set.csv"))
classes = [str(c) for c in eng.label_encoder.classes_]
name2idx = {c.lower(): i for i, c in enumerate(classes)}
NORMAL_IDX = name2idx["normal"]
n_classes = len(classes)

F = ["dur","proto","service","state","spkts","dpkts","sbytes","dbytes","rate","sttl","dttl",
 "sload","dload","sloss","dloss","sinpkt","dinpkt","sjit","djit","swin","stcpb","dtcpb","dwin",
 "tcprtt","synack","ackdat","smean","dmean","trans_depth","response_body_len","ct_srv_src",
 "ct_state_ttl","ct_dst_ltm","ct_src_dport_ltm","ct_dst_sport_ltm","ct_dst_src_ltm",
 "is_ftp_login","ct_ftp_cmd","ct_flw_http_mthd","ct_src_ltm","ct_srv_dst","is_sm_ips_ports"]
raw = df[[c for c in F if c in df.columns]].copy()
flow = eng._prepare_flow(raw)

yt_cat = np.array([name2idx.get(s.lower(), NORMAL_IDX) for s in df["attack_cat"].astype(str)])
yt_bin = df["label"].astype(int).values
n = len(yt_bin)
print(f"test set: {n:,} rows")

In [ ]:
# ------------------------------------------------------------
# 2. LAYER 4 -- reuse the production autoencoder unchanged (it was always
# trained on UNSW-only Normal traffic, CICIDS never touched it, so it needs
# no retrain). Its expected input scale is the PRODUCTION preprocessor's,
# not the new UNSW-only one -- keep the two preprocessors separate.
# ------------------------------------------------------------
X_old = eng.preprocessor.transform(flow)
ae_thresh = eng.anomaly_config["autoencoder"]["operational_threshold"]
recon = eng.autoencoder.predict(X_old, verbose=0)
ae_err = np.mean((X_old - recon) ** 2, axis=1)
print(f"ae_thresh: {ae_thresh:.5f}")

In [ ]:
# ------------------------------------------------------------
# 3. LAYERS 2+3 -- the new UNSW-only models, scored with the new preprocessor
# ------------------------------------------------------------
unsw_preprocessor = joblib.load(os.path.join(NB_OUT, "preprocessor.pkl"))
unsw_label_encoder = joblib.load(os.path.join(NB_OUT, "label_encoder.pkl"))
assert list(unsw_label_encoder.classes_) == classes, "label encoding mismatch between production and UNSW-only models"

X_new = unsw_preprocessor.transform(flow)

rf_unsw = joblib.load(os.path.join(NB_OUT, "models", "randomforest.pkl"))
xgb_unsw = joblib.load(os.path.join(NB_OUT, "models", "xgboost.pkl"))
cat_unsw = joblib.load(os.path.join(NB_OUT, "models", "catboost.pkl"))
dnn_unsw = keras.models.load_model(os.path.join(NB_OUT, "models", "dnn_specialist.keras"))

p_rf = rf_unsw.predict_proba(X_new)
p_xgb = xgb_unsw.predict_proba(X_new)
p_cat = cat_unsw.predict_proba(X_new)
p_dl = np.array(dnn_unsw.predict(X_new, verbose=0))
print("p_rf", p_rf.shape, "p_xgb", p_xgb.shape, "p_cat", p_cat.shape, "p_dl", p_dl.shape)

In [ ]:
# ------------------------------------------------------------
# 4. CACHE RAW PER-ROW PREDICTIONS (same schema as arxiv_paper/results/cached_predictions.npz)
# ------------------------------------------------------------
cache_path = os.path.join(RESULTS_OUT, "unsw_only_cached_predictions.npz")
np.savez_compressed(cache_path, p_rf=p_rf, p_xgb=p_xgb, p_cat=p_cat, p_dl=p_dl,
                     ae_err=ae_err, ae_thresh=ae_thresh, yt_bin=yt_bin, yt_cat=yt_cat,
                     classes=np.array(classes, dtype=object))
print(f"WROTE {cache_path}")

In [ ]:
# ------------------------------------------------------------
# 5. FUSION CASCADE -- identical logic to ablation_study.py / day1_multiclass_analysis.py,
# DNN threshold fixed at 0.50 (current production value)
# ------------------------------------------------------------
DL_THRESHOLD = 0.50

def fuse(l2=True, l3=True, l4=True):
    fb = np.zeros(n, dtype=int)
    fc = np.empty(n, dtype=int)
    for i in range(n):
        decided = None
        if l2:
            experts = []
            for P in (p_rf, p_xgb, p_cat):
                idx = int(np.argmax(P[i]))
                experts.append((idx, float(P[i][idx])))
            mal = [(c, cf) for c, cf in experts if c != NORMAL_IDX and cf >= 0.70]
            if len(mal) < 2:
                soft = [(c, cf) for c, cf in experts if c != NORMAL_IDX and cf >= 0.60]
                if len(soft) >= 2 and [c for c, _ in soft].count(soft[0][0]) >= 2:
                    mal = [(c, cf) for c, cf in soft if c == soft[0][0]]
            if len(mal) >= 2 or any(cf >= 0.92 for _, cf in mal):
                decided = max(mal, key=lambda x: x[1])[0]
        if decided is None and l3:
            di = int(np.argmax(p_dl[i]))
            if di != NORMAL_IDX and float(p_dl[i][di]) >= DL_THRESHOLD:
                decided = di
        if decided is not None:
            fc[i] = decided
            fb[i] = 1
        elif l4 and ae_err[i] > ae_thresh:
            fc[i] = n_classes
            fb[i] = 1
        else:
            fc[i] = NORMAL_IDX
            fb[i] = 0
    return fb, fc

In [ ]:
# ------------------------------------------------------------
# 6. BOOTSTRAP CI + MCNEMAR -- same exact vectorised formulas validated in
# day1_statistical_rigor.py (row-level bootstrap resampling of a confusion
# matrix is exactly equivalent to multinomial resampling over its cells)
# ------------------------------------------------------------
RNG = np.random.default_rng(42)
N_BOOT = 1000

def ci_from_samples(v):
    return {"ci95_low": round(float(np.percentile(v, 2.5)), 4),
            "ci95_high": round(float(np.percentile(v, 97.5)), 4)}

def bootstrap_ci_binary(fb, y_true=yt_bin, n_boot=N_BOOT):
    cat = y_true.astype(int) * 2 + fb.astype(int)
    counts = np.bincount(cat, minlength=4)
    probs = counts / n
    draws = RNG.multinomial(n, probs, size=n_boot).astype(float)
    TN, FP, FN, TP = draws[:, 0], draws[:, 1], draws[:, 2], draws[:, 3]
    acc = (TP + TN) / n
    prec = np.divide(TP, TP + FP, out=np.zeros(n_boot), where=(TP + FP) > 0)
    rec = np.divide(TP, TP + FN, out=np.zeros(n_boot), where=(TP + FN) > 0)
    f1 = np.divide(2 * prec * rec, prec + rec, out=np.zeros(n_boot), where=(prec + rec) > 0)
    return {"accuracy": ci_from_samples(acc), "precision": ci_from_samples(prec),
            "recall": ci_from_samples(rec), "f1": ci_from_samples(f1)}

def bootstrap_ci_multiclass(pred, y_true=yt_cat, n_boot=N_BOOT):
    n_pred = n_classes + 1
    cell = y_true.astype(int) * n_pred + pred.astype(int)
    counts = np.bincount(cell, minlength=n_classes * n_pred)
    probs = counts / n
    draws = RNG.multinomial(n, probs, size=n_boot).astype(float).reshape(n_boot, n_classes, n_pred)
    TP = np.stack([draws[:, k, k] for k in range(n_classes)], axis=1)
    support = draws.sum(axis=2)
    pred_total = draws[:, :, :n_classes].sum(axis=1)
    FN = support - TP
    FP = pred_total - TP
    precision = np.divide(TP, TP + FP, out=np.zeros_like(TP), where=(TP + FP) > 0)
    recall = np.divide(TP, TP + FN, out=np.zeros_like(TP), where=(TP + FN) > 0)
    f1 = np.divide(2 * precision * recall, precision + recall,
                    out=np.zeros_like(TP), where=(precision + recall) > 0)
    acc = TP.sum(axis=1) / n
    f1_macro = f1.mean(axis=1)
    f1_weighted = (f1 * support).sum(axis=1) / support.sum(axis=1)
    return {"accuracy": ci_from_samples(acc), "f1_macro": ci_from_samples(f1_macro),
            "f1_weighted": ci_from_samples(f1_weighted)}

def mcnemar_test(correct_a, correct_b):
    b = int(np.sum(correct_a & ~correct_b))
    c = int(np.sum(~correct_a & correct_b))
    n_disc = b + c
    if n_disc == 0:
        return {"b": b, "c": c, "n_discordant": 0, "p_value": 1.0, "method": "no discordant pairs"}
    if n_disc < 25:
        res = stats.binomtest(min(b, c), n_disc, 0.5)
        return {"b": b, "c": c, "n_discordant": n_disc, "p_value": round(float(res.pvalue), 6),
                "method": "exact binomial"}
    stat = (abs(b - c) - 1) ** 2 / (b + c)
    p = 1 - stats.chi2.cdf(stat, df=1)
    return {"b": b, "c": c, "n_discordant": n_disc, "statistic": round(float(stat), 4),
            "p_value": round(float(p), 6), "method": "chi-square (continuity-corrected)"}

In [ ]:
# ------------------------------------------------------------
# 7. BINARY ABLATION, threshold=0.50 -- directly comparable to Table 7
# (the CICIDS-mixed production model at the same threshold)
# ------------------------------------------------------------
configs = {}
for name, flags in [("full_system", dict(l2=True, l3=True, l4=True)),
                     ("no_L2", dict(l2=False, l3=True, l4=True)),
                     ("no_L3", dict(l2=True, l3=False, l4=True)),
                     ("no_L4", dict(l2=True, l3=True, l4=False))]:
    fb, fc = fuse(**flags)
    configs[name] = {"fb": fb, "fc": fc}

binary_results = {}
header = f'{"Configuration":<14}{"Accuracy":>10}{"Precision":>11}{"Recall":>9}{"F1":>9}'
print(header)
for name, c in configs.items():
    fb = c["fb"]
    m = {"accuracy": round(float(accuracy_score(yt_bin, fb)), 4),
         "precision": round(float(precision_score(yt_bin, fb, zero_division=0)), 4),
         "recall": round(float(recall_score(yt_bin, fb, zero_division=0)), 4),
         "f1": round(float(f1_score(yt_bin, fb, zero_division=0)), 4)}
    m["ci95"] = bootstrap_ci_binary(fb)
    binary_results[name] = m
    acc, prec, rec, f1v = m['accuracy'], m['precision'], m['recall'], m['f1']
    print(f'{name:<14}{acc:>10.4f}{prec:>11.4f}{rec:>9.4f}{f1v:>9.4f}')

full_correct = configs["full_system"]["fb"] == yt_bin
mcnemar_binary = {}
for name in ["no_L2", "no_L3", "no_L4"]:
    other_correct = configs[name]["fb"] == yt_bin
    key = f'full_vs_{name}'
    mcnemar_binary[key] = mcnemar_test(full_correct, other_correct)
    print(f'  full vs {name}: p={mcnemar_binary[key]["p_value"]}')

In [ ]:
# ------------------------------------------------------------
# 8. MULTICLASS -- per-model standalone and fused-system ablation, directly
# comparable to Tables 1 and 3
# ------------------------------------------------------------
def mc_metrics(y_true, y_pred, labels):
    return {"accuracy": round(float(accuracy_score(y_true, y_pred)), 4),
            "f1_macro": round(float(f1_score(y_true, y_pred, average="macro", labels=labels, zero_division=0)), 4),
            "f1_weighted": round(float(f1_score(y_true, y_pred, average="weighted", labels=labels, zero_division=0)), 4)}

labels = list(range(n_classes))
per_model_results = {}
for name, P in [("RandomForest", p_rf), ("XGBoost", p_xgb), ("CatBoost", p_cat), ("DNN_Specialist", p_dl)]:
    pred = np.argmax(P, axis=1)
    m = mc_metrics(yt_cat, pred, labels)
    m["ci95"] = bootstrap_ci_multiclass(pred)
    per_model_results[name] = m
    print(f'{name:<16} acc={m["accuracy"]:.4f} f1_macro={m["f1_macro"]:.4f}')

fused_mc_results = {}
for name in ["full_system", "no_L2", "no_L3", "no_L4"]:
    fc = configs[name]["fc"]
    m = mc_metrics(yt_cat, fc, labels)
    m["ci95"] = bootstrap_ci_multiclass(fc)
    fused_mc_results[name] = m
    print(f'{name:<16} acc={m["accuracy"]:.4f} f1_macro={m["f1_macro"]:.4f}')

In [ ]:
# ------------------------------------------------------------
# 9. SIDE BY SIDE -- UNSW-only vs. the published CICIDS-mixed production numbers,
# same 175,341-row test set, same threshold, same fusion cascade
# ------------------------------------------------------------
published_binary_050 = {
    "full_system": 0.8548, "no_L2": 0.8446, "no_L3": 0.8539, "no_L4": 0.7617,
}
published_per_model = {
    "RandomForest": 0.8092, "XGBoost": 0.8280, "CatBoost": 0.6997, "DNN_Specialist": 0.7052,
}
published_fused_mc = {
    "full_system": 0.7047, "no_L2": 0.6437, "no_L3": 0.7307, "no_L4": 0.7097,
}

header2 = f'{"":20}{"CICIDS-mixed":>14}{"UNSW-only":>12}{"delta":>9}'
print()
print(header2)
print("--- binary accuracy, threshold=0.50 ---")
for name in binary_results:
    pub = published_binary_050[name]
    new = binary_results[name]["accuracy"]
    print(f"{name:<20}{pub:>14.4f}{new:>12.4f}{new-pub:>+9.4f}")
print("--- per-model multiclass accuracy ---")
for name in per_model_results:
    pub = published_per_model[name]
    new = per_model_results[name]["accuracy"]
    print(f"{name:<20}{pub:>14.4f}{new:>12.4f}{new-pub:>+9.4f}")
print("--- fused multiclass accuracy, threshold=0.50 ---")
for name in fused_mc_results:
    pub = published_fused_mc[name]
    new = fused_mc_results[name]["accuracy"]
    print(f"{name:<20}{pub:>14.4f}{new:>12.4f}{new-pub:>+9.4f}")

In [ ]:
# ------------------------------------------------------------
# 10. SAVE CONSOLIDATED RESULTS
# ------------------------------------------------------------
out_path = os.path.join(RESULTS_OUT, "unsw_only_ablation_comparison.json")
results = {
    "test_samples": n,
    "classes": classes,
    "dl_threshold": DL_THRESHOLD,
    "n_bootstrap": N_BOOT,
    "binary_050": binary_results,
    "mcnemar_binary_vs_full": mcnemar_binary,
    "multiclass_per_model": per_model_results,
    "multiclass_fused": fused_mc_results,
    "published_reference": {
        "binary_050": published_binary_050,
        "per_model": published_per_model,
        "fused_multiclass": published_fused_mc,
    },
}
json.dump(results, open(out_path, "w"), indent=2)
print(f"\nWROTE {out_path}")